In [1]:
import os
import sys
sys.path.append("..")
import nibabel as nib
import matplotlib.pyplot as plt
#from ipywidgets import interact
import numpy as np
#import ipywidgets as widgets
import pandas as pd
from tqdm import tqdm
import csv
import numpy as np
import random

import glob
#import torch
#from monai import transforms
#import utils.custom_transforms as custom_transforms
#from monai.data import CacheDataset, DataLoader, ThreadDataLoader


In [11]:
#ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
#ROOT_DIR = "/home/theotime/bettik/"
ROOT_DIR = "/home/fehrdelt/bettik/"

In [30]:

final_adc_dataset_small_HCP_YA = ROOT_DIR+"datasets/final_adc_dataset_small/HCP-YA_registered/"
final_adc_dataset_small_Dallas = ROOT_DIR+"datasets/final_adc_dataset_small/Dallas_registered/"
final_adc_dataset_small_AINI_stroke_ait = ROOT_DIR+"datasets/final_adc_dataset_small/AIT_final_registered/"
final_adc_dataset_small_ixi = ROOT_DIR+"datasets/final_adc_dataset_small/ixi_registered/"
final_adc_dataset_small_isles = ROOT_DIR+"datasets/final_adc_dataset_small/ISLES_registered/"

final_flair_dataset_small_dallas = ROOT_DIR+"datasets/final_flair_dataset_small/dallas_registered/"
final_flair_dataset_small_lemon = ROOT_DIR+"datasets/final_flair_dataset_small/lemon_registered/"
final_flair_dataset_small_oasis = ROOT_DIR+"datasets/final_flair_dataset_small/oasis_registered/"

final_flair_dataset_small_isles = ROOT_DIR+"datasets/final_flair_dataset_small/isles_registered/"
final_flair_dataset_small_brats = ROOT_DIR+"datasets/final_flair_dataset_small/brats_registered/"

final_t1_dataset_small_ixi = ROOT_DIR+"datasets/final_t1_dataset_small/ixi_registered/"
final_t1_dataset_small_oasis = ROOT_DIR+"datasets/final_t1_dataset_small/oasis_registered/"
final_t1_dataset_small_dallas = ROOT_DIR+"datasets/final_t1_dataset_small/dallas_registered/"

final_soop_dataset_small_flair = ROOT_DIR+"datasets/final_soop_dataset_small/flair_registered/"
final_soop_dataset_small_adc = ROOT_DIR+"datasets/final_soop_dataset_small/adc_registered/"

final_aini_stroke_dataset_small_adc = ROOT_DIR+"datasets/final_aini_stroke_dataset_small/adc_registered/"
final_aini_stroke_dataset_small_flair = ROOT_DIR+"datasets/final_aini_stroke_dataset_small/flair_registered/"

### Display middle slice of every nifti file in folder

In [13]:
def display_middle_slice(folder_path, axe=2, nb_images=49, page_number=0, figsize=25, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=3000):
    # List all files in the folder
    files = os.listdir(folder_path)[nb_images*page_number:nb_images*(page_number+1)]

    # Filter out non-NIFTI files
    nifti_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    
    # Create a figure to display all middle slices and their histograms
    rows = int(np.sqrt(nb_images))
    fig, axes = plt.subplots(rows, rows, figsize=(figsize, figsize+2))
    plt.tight_layout()

    for i, nifti_file in enumerate(nifti_files):
        # Load the NIFTI image
        img_path = os.path.join(folder_path, nifti_file)
        img = nib.load(img_path)
        data = img.get_fdata()

        # Get the middle slice along the specified axis
        if axe == 0:
            middle_slice = data.shape[0] // 2
            image_slice = data[middle_slice, :, :]
        elif axe == 1:
            middle_slice = data.shape[1] // 2
            image_slice = data[:, middle_slice, :]
        elif axe==2:
            middle_slice = data.shape[2] // 2
            image_slice = data[:, :, middle_slice]


        if normalize_from_histogram_peak:
            # Compute the histogram of the image slice
            hist, bins = np.histogram(image_slice.flatten(), bins=100, range=(np.max(image_slice)/5.0, np.max(image_slice)))

            # Find the value corresponding to the maximum of the histogram
            most_occurred_pixel_value = bins[np.argmax(hist)]

            image_slice = image_slice/most_occurred_pixel_value*hist_norm_target_value # scale it so the peak is always at hist_norm_target_value

        # Display the middle slice for the z-axis
        if normalize_from_histogram_peak:
            axes[i//rows, (i%rows)].imshow(image_slice, cmap='gray', vmin=0, vmax=max_display_value) # flair: vmax=450
        else:
            axes[i//rows, (i%rows)].imshow(image_slice, cmap='gray', vmin=0, vmax=np.max(image_slice))
        
        axes[i//rows, (i%rows)].set_title(f'{i+(page_number*nb_images)}')
        axes[i//rows, (i%rows)].axis("off")

        axes[i//rows, (i%rows)].set_aspect('auto') # Set the aspect ratio to auto to match the imshow plot
        axes[i//rows, (i%rows)].set_box_aspect(1)  # Set the aspect ratio of the histogram subplot


    # Set the title of the figure
    fig.suptitle(f'Middle slices and histograms for all NIFTI files, axe={axe}, page {page_number}', fontsize=16)
    
    # Show the plot
    plt.show()


In [14]:
from multiprocessing import Pool, cpu_count

def _process_single_image(args):
    """Helper function to process a single image in parallel"""
    folder_path, nifti_file, axe, normalize_from_histogram_peak, hist_norm_target_value = args
    
    # Load the NIFTI image
    img_path = os.path.join(folder_path, nifti_file)
    try:
        img = nib.load(img_path)
        data = img.get_fdata()
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return np.zeros((256,256))  # Return a dummy array in case of error to avoid breaking the loop
    

    # Get the middle slice along the specified axis
    if axe == 0:
        middle_slice = data.shape[0] // 2
        image_slice = data[middle_slice, :, :]
    elif axe == 1:
        middle_slice = data.shape[1] // 2
        image_slice = data[:, middle_slice, :]
    elif axe == 2:
        middle_slice = data.shape[2] // 2
        image_slice = data[:, :, middle_slice]

    if normalize_from_histogram_peak:
        # Compute the histogram of the image slice
        hist, bins = np.histogram(image_slice.flatten(), bins=100, range=(np.max(image_slice)/5.0, np.max(image_slice)))
        # Find the value corresponding to the maximum of the histogram
        most_occurred_pixel_value = bins[np.argmax(hist)]
        image_slice = image_slice/most_occurred_pixel_value*hist_norm_target_value
    
    return image_slice

def display_middle_slice_multiprocessing(folder_path, axe=2, nb_images=49, page_number=0, figsize=25, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=3000):
    # List all files in the folder
    files = os.listdir(folder_path)[nb_images*page_number:nb_images*(page_number+1)]

    # Filter out non-NIFTI files
    nifti_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    
    # Prepare arguments for parallel processing
    args_list = [(folder_path, nifti_file, axe, normalize_from_histogram_peak, hist_norm_target_value) 
                 for nifti_file in nifti_files]
    
    # Process images in parallel
    num_processes = min(cpu_count(), len(nifti_files))
    with Pool(processes=num_processes) as pool:
        image_slices = pool.map(_process_single_image, args_list)
    
    # Create a figure to display all middle slices
    rows = int(np.sqrt(nb_images))
    fig, axes = plt.subplots(rows, rows, figsize=(figsize, figsize+2))
    plt.tight_layout()

    for i, image_slice in enumerate(image_slices):
        #if (i+(page_number*nb_images)) not in failed_registration_final_t1_dataset_oasis:
        if True:
            # Display the middle slice
            if normalize_from_histogram_peak:
                axes[i//rows, (i%rows)].imshow(image_slice, cmap='gray', vmin=0, vmax=max_display_value)
            else:
                axes[i//rows, (i%rows)].imshow(image_slice, cmap='gray', vmin=0, vmax=np.max(image_slice))
            
            axes[i//rows, (i%rows)].set_title(f'{i+(page_number*nb_images)}')
            axes[i//rows, (i%rows)].axis("off")
            axes[i//rows, (i%rows)].set_aspect('auto')
            axes[i//rows, (i%rows)].set_box_aspect(1)

    # Set the title of the figure
    fig.suptitle(f'Middle slices and histograms for all NIFTI files, axe={axe}, page {page_number}', fontsize=16)
    
    # Show the plot
    plt.show()


In [15]:
def display_middle_slice_z_with_histogram(folder_path, nb_images, page_number, figsize, normalize_from_histogram_peak=True, normalize_before_select_slice = True, hist_norm_target_value=200, max_display_value=3000):
    # List all files in the folder
    files = os.listdir(folder_path)[nb_images*page_number:nb_images*(page_number+1)]

    # Filter out non-NIFTI files
    nifti_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    
    # Create a figure to display all middle slices and their histograms
    rows = int(np.sqrt(nb_images))
    fig, axes = plt.subplots(rows, rows * 2, figsize=(figsize, figsize//2 + 2))
    plt.tight_layout()

    for i, nifti_file in enumerate(nifti_files):
        # Load the NIFTI image
        img_path = os.path.join(folder_path, nifti_file)
        img = nib.load(img_path)
        data = img.get_fdata()

        if normalize_before_select_slice and normalize_from_histogram_peak:
            # Compute the histogram of the image slice
            hist, bins = np.histogram(data.flatten(), bins=100, range=(np.max(data)/15.0, np.max(data)))
            #print(f"{i+(page_number*nb_images)}: max={np.max(data)}, max/5={np.max(data)/5.0}")
            # Find the value corresponding to the maximum of the histogram
            most_occurred_pixel_value = bins[np.argmax(hist)]

            data = data/most_occurred_pixel_value*hist_norm_target_value # scale it so the peak is always at hist_norm_target_value

            # Get the middle slice index for the z-axis
            middle_slice_z = data.shape[2] // 2

            image_slice = data[:, :, middle_slice_z]

        elif not normalize_before_select_slice and normalize_from_histogram_peak:

            # Get the middle slice index for the z-axis
            middle_slice_z = data.shape[2] // 2

            image_slice = data[:, :, middle_slice_z]

            # Compute the histogram of the image slice
            hist, bins = np.histogram(image_slice.flatten(), bins=100, range=(np.max(image_slice)/5.0, np.max(image_slice)))

            # Find the value corresponding to the maximum of the histogram
            most_occurred_pixel_value = bins[np.argmax(hist)]

            image_slice = image_slice/most_occurred_pixel_value*hist_norm_target_value # scale it so the peak is always at hist_norm_target_value

        # Display the middle slice for the z-axis
        if normalize_from_histogram_peak:
            axes[i//rows, (i%rows)*2].imshow(image_slice, cmap='gray', vmin=0, vmax=max_display_value) # flair: vmax=450
        else:
            axes[i//rows, (i%rows)*2].imshow(image_slice, cmap='gray', vmin=0, vmax=np.max(image_slice))
        axes[i//rows, (i%rows)*2].set_title(f'{i+(page_number*nb_images)}')
        axes[i//rows, (i%rows)*2].axis("off")

        axes[i//rows, (i%rows)*2].set_aspect('auto') # Set the aspect ratio to auto to match the imshow plot
        axes[i//rows, (i%rows)*2].set_box_aspect(1)  # Set the aspect ratio of the histogram subplot


        #hist_values, bin_edges, _ = axes[i//rows, (i%rows)*2 + 1].hist(image_slice.flatten(), bins=100, range=(10,max_display_value)) #flair: range=(1, 450)
        hist_values, bin_edges, _ = axes[i//rows, (i%rows)*2 + 1].hist(data.flatten(), bins=100, range=(10,max_display_value)) #flair: range=(1, 450)
        axes[i//rows, (i%rows)*2 + 1].axvline(x=hist_norm_target_value, color='r', linestyle='--', label=f'Target ({hist_norm_target_value})')
        axes[i//rows, (i%rows)*2 + 1].legend()
        axes[i//rows, (i%rows)*2 + 1].set_box_aspect(1)  # Set the aspect ratio of the histogram subplot

    # Set the title of the figure
    fig.suptitle('Middle slices and histograms for all NIFTI files')
    
    # Show the plot
    plt.show()


### final_t1_dataset_small

### IXI


In [31]:
final_t1_dataset_small_ixi_to_exclude = [15, 51, 95, 180, 266, 275, 488, 561]
final_t1_dataset_small_oasis_to_exclude = []

In [32]:
display_middle_slice_multiprocessing(final_t1_dataset_small_oasis, axe=2, nb_images=64, page_number=0, figsize=15, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=400)


ValueError: Number of processes must be at least 1

In [27]:
# Get the list of files from the aini stroke ADC dataset

ixi_t1_files = os.listdir(final_t1_dataset_small_ixi)

failed_registration_ixi_t1_dataset_small = []

# Print the filenames corresponding to the failed registration indexes
print(f"Failed registration files in final_t1_dataset_ixi ({len(failed_registration_ixi_t1_dataset_small)} files):\n")

for idx in final_t1_dataset_small_ixi_to_exclude:
    if idx < len(ixi_t1_files):
        print(f"Index {idx}: {ixi_t1_files[idx]}")
        failed_registration_ixi_t1_dataset_small.append(ixi_t1_files[idx])
    else:
        print(f"Index {idx}: OUT OF RANGE (total files: {len(ixi_t1_files)})")



Failed registration files in final_t1_dataset_ixi (0 files):

Index 15: IXI640-Guys-1106-T1.nii.gz
Index 51: IXI285-Guys-0857-T1.nii.gz
Index 95: IXI639-Guys-1088-T1.nii.gz
Index 180: IXI090-Guys-0800-T1.nii.gz
Index 266: IXI211-HH-1568-T1.nii.gz
Index 275: IXI636-HH-2733-T1.nii.gz
Index 488: IXI102-HH-1416-T1.nii.gz
Index 561: IXI489-Guys-1014-T1.nii.gz


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for i, filename in enumerate(failed_registration_ixi_t1_dataset_small):
    img_path = os.path.join(final_t1_dataset_small_ixi, filename)
    
    # Load the NIFTI image
    img = nib.load(img_path)
    data = img.get_fdata()
    
    # Get the middle axial slice
    mid_z = data.shape[2] // 2
    image_slice = data[:, :, mid_z]
    
    # Display the slice
    axes[i].imshow(image_slice, cmap='gray')
    axes[i].set_title(filename, fontsize=10)
    axes[i].axis('off')

# Hide any remaining blank subplots if any
for j in range(len(failed_registration_ixi_t1_dataset_small), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

### Aini-stroke ADC

In [19]:
failed_aini_stroke_adc = [15]

In [ ]:
display_middle_slice_multiprocessing(final_aini_stroke_dataset_small_adc, axe=2, nb_images=64, page_number=0, figsize=15, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=400)


In [20]:
# Get the list of files from the aini stroke ADC dataset

adc_files = os.listdir(final_aini_stroke_dataset_small_adc)

failed_registration_final_aini_stroke_dataset_small_adc_filenames = []

# Print the filenames corresponding to the failed registration indexes
print(f"Failed registration files in aini stroke ADC dataset ({len(failed_registration_final_aini_stroke_dataset_small_adc_filenames)} files):\n")

for idx in failed_aini_stroke_adc:
    if idx < len(adc_files):
        print(f"Index {idx}: {adc_files[idx]}")
        failed_registration_final_aini_stroke_dataset_small_adc_filenames.append(adc_files[idx])
    else:
        print(f"Index {idx}: OUT OF RANGE (total files: {len(adc_files)})")



Failed registration files in aini stroke ADC dataset (0 files):

Index 15: aini-stroke-22629.nii.gz


In [ ]:
filename = "aini-stroke-22629.nii.gz"
img_path = os.path.join(final_aini_stroke_dataset_small_adc, filename)

img = nib.load(img_path)
data = img.get_fdata()

mid_z = data.shape[2] // 2
slice_img = data[:, :, mid_z]

plt.figure(figsize=(5, 5))
plt.imshow(slice_img, cmap="gray")
plt.title(f"Middle axial slice: {filename}")
plt.axis("off")
plt.show()

### Aini-stroke FLAIR

In [22]:
failed_aini_stroke_flair = [3, 15, 24, 30, 41, 73, 89, 128, 144, 181, ]

In [ ]:
display_middle_slice_multiprocessing(final_aini_stroke_dataset_small_flair, axe=2, nb_images=64, page_number=3, figsize=15, normalize_from_histogram_peak=False, hist_norm_target_value=200, max_display_value=400)


In [23]:
# Get the list of files from the aini stroke FLAIR dataset

flair_files = os.listdir(final_aini_stroke_dataset_small_flair)

failed_registration_final_aini_stroke_dataset_small_flair_filenames = []

# Print the filenames corresponding to the failed registration indexes
print(f"Failed registration files in aini stroke flair dataset ({len(failed_registration_final_aini_stroke_dataset_small_flair_filenames)} files):\n")

for idx in failed_aini_stroke_flair:
    if idx < len(flair_files):
        print(f"Index {idx}: {flair_files[idx]}")
        failed_registration_final_aini_stroke_dataset_small_flair_filenames.append(flair_files[idx])
    else:
        print(f"Index {idx}: OUT OF RANGE (total files: {len(flair_files)})")



Failed registration files in aini stroke flair dataset (0 files):

Index 3: aini-stroke-22311.nii.gz
Index 15: aini-stroke-22666.nii.gz
Index 24: aini-stroke-22691.nii.gz
Index 30: aini-stroke-22162.nii.gz
Index 41: aini-stroke-22486.nii.gz
Index 73: aini-stroke-22287.nii.gz
Index 89: aini-stroke-22821.nii.gz
Index 128: aini-stroke-22243.nii.gz
Index 144: aini-stroke-22232.nii.gz
Index 181: aini-stroke-22468.nii.gz


In [ ]:
filename = "aini-stroke-22311.nii.gz"
img_path = os.path.join(final_aini_stroke_dataset_small_flair, filename)

img = nib.load(img_path)
data = img.get_fdata()

mid_z = data.shape[2] // 2
slice_img = data[:, :, mid_z]

plt.figure(figsize=(5, 5))
plt.imshow(slice_img, cmap="gray")
plt.title(f"Middle axial slice: {filename}")
plt.axis("off")
plt.show()

#### T1
256x256x256

In [ ]:
failed_t1_small_dallas = [16, 27, 62, 66, 79, 88, 162, 163, 243, 249, 260, 338, 358, 364, 470, 483, 592, 600, 708, 718, 731, 770, 797, 818, 834, 867, 887, 893, 944]

In [ ]:
display_middle_slice_multiprocessing(final_adc_dataset_small_Dallas, axe=2, nb_images=64, page_number=3, figsize=15, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=400)

In [76]:
failed_registration_final_t1_dataset_ixi = [50, 175, 197]
failed_registration_final_t1_dataset_oasis = [
2,
3,
10,
11,
13,
24,
31,
39,
45,
49,
54,
62,
65,
67,
79,
84,
88,
93,
100,
104,
111,
122,
128,
129,
131,
133,
135,
141,
144,
145,
147,
150,
151,
152,
156,
165,
169,
170,
173,
175,
211,
217,
226,
229,
237,
243,
249,
250,
257,
259,
262,
263,
271,
272,
284,
290,
294,
310,
312,
314,
318,
341,
344,
353,
382,
387,
392,
401,
409,
411,
412,
417,
419,
424,
430,
434,
436,
439,
440,
457,
458,
461,
462,
464,
471,
476,
479,
481,
487,
493,
501,
507,
508,
518,
523,
531,
535,
550,
554,
558,
565,
569,
583,
584,
585,
596,
602,
609,
613,
616,
633,
646,
649,
658,
670,
676,
678,
683,
695,
702,
705,
711,
716,
721,
724,
731,
735,
740,
745,
746,
754,
762,
765,
769,
777,
779,
783,
785,
791,
797,
818,
824,
829,
831,
745,
845,
847,
852,
857,
861,
862,
866,
867,
868,
869,
874,
876,
884,
902,
904,
907,
913,
916,
931,
933,
936,
946,
950,
966,
970,
977,
982,
992,
993,
997,
998,
999,
1007,
1008,
1012,
1014,
1021,
1024,
1032,
1034,
1040,
1047,
1054,
1065,
1067,
1069,
1079,
1085,
1087,
1091,
1094,
1097,
1101,
1102,
1103,
1107,
1108,
1122,
1135,
1146,
1148,
1155,
1159,
1170,
1174,
1175,
1177,
1183,
1185,
1186,
1188,
1198,
1199,
1201,
1206,
1212,
1218,
1220,
1221,
1234,
1237,
1241,
1244,
1274,
1247,
1248,
1249,
1253,
1254,
1255,
1256,
1258,
1260,
1264,
1265,
1269,
1277,
1279,
1280,
1286,
1292,
1301,
1303,
1312,
1320,
1321,
1325,
1337,
1345,
1348,
1350,
1353,
1361,
1370,
1373,
1386,
1390,
1397,
1407,
1406,
1408,
1410,
1414,
1433,
1416,
1423,
1425,
1434,
1440,
1442,
1456,
1457,
1463,
1464,
1474,
1481,
1484,
1487,
1500,
1507,
1525,
1527,
1528,
1538,
1544,
1552,
1557,
1558,
1559,
1562,
1564,
1565,
1575,
1580,
1582,
1583,
1585,
1589,
1594,
1601,
1604,
1609,
1615,
1617,
1620,
1621,
1624,
1633,
1636,
1641,
1646,
1647,
1650,
1651,
1661,
1666,
1681,
1693,
1694,
1696,
1698,
1702,
1707,
1720,
1722,
1735,
1737,
1741,
1752,
1774,
1775,
1776,
1786,
1789,
1790,
1792,
1800,
1806,
1809,
1817,
1824,
1835,
1841,
1850,
1855,
1859,
1860,
1865,
1873,
1881,
1904,
1908,
1909,
1912,
1925,
1939,
1941,
1942,
1951,
1959,
1963,
1966,
1963,
1966,
1969,
1970,
1976,
1981,
1987,
1996,
1998,
2010,
2011,
2013,
2019,
2020,
2022,
2023,
2024,
2032,
2036,
2053,
2062,
2063,
2065,
2071,
2078,
2083,
2087,
2098,
2104,
2108,
2118,
2123,
2129,
2130,
2139,
2149,
2153,
2157,
2158,
2165,
2166,
2167,
2185,
2188,
2189,
2195,
2197,
2207,
2215,
2228,
2234,
2237,
2238,
2241,
2256,
2259,
2260,
2262,
2265,
2267,
2280,
2282,
2286,
2289,
2290,
2296,
2314,
2315,
2319,
2328,
2332,
2343,
2346,
2347,
2353,
2354
]

In [69]:
# Get the list of files from the OASIS T1 dataset
oasis_files = os.listdir(final_t1_dataset_oasis)
failed_registration_final_t1_dataset_oasis_filenames = []

# Print the filenames corresponding to the failed registration indexes
print(f"Failed registration files in OASIS T1 dataset ({len(failed_registration_final_t1_dataset_oasis)} files):\n")
for idx in failed_registration_final_t1_dataset_oasis:
    if idx < len(oasis_files):
        print(f"Index {idx}: {oasis_files[idx]}")
        failed_registration_final_t1_dataset_oasis_filenames.append(oasis_files[idx])
    else:
        print(f"Index {idx}: OUT OF RANGE (total files: {len(oasis_files)})")



Failed registration files in OASIS T1 dataset (443 files):

Index 2: sub-OAS30094_ses-d0679_run-02_T1w_brain_anat_n4.nii.gz
Index 3: sub-OAS30699_ses-d0108_run-03_T1w_brain_anat_n4.nii.gz
Index 10: sub-OAS30259_ses-d0000_run-01_T1w_brain_anat_n4.nii.gz
Index 11: sub-OAS30203_ses-d0802_run-04_T1w_brain_anat_n4.nii.gz
Index 13: sub-OAS30144_ses-d1204_run-01_T1w_brain_anat_n4.nii.gz
Index 24: sub-OAS30662_ses-d1483_run-02_T1w_brain_anat_n4.nii.gz
Index 31: sub-OAS30470_ses-d1023_run-03_T1w_brain_anat_n4.nii.gz
Index 39: sub-OAS31132_ses-d3392_run-01_T1w_brain_anat_n4.nii.gz
Index 45: sub-OAS31064_ses-d0785_run-02_T1w_brain_anat_n4.nii.gz
Index 49: sub-OAS30048_ses-d0983_run-02_T1w_brain_anat_n4.nii.gz
Index 54: sub-OAS30860_ses-d0134_run-02_T1w_brain_anat_n4.nii.gz
Index 62: sub-OAS31122_ses-d2180_run-03_T1w_brain_anat_n4.nii.gz
Index 65: sub-OAS30278_ses-d0028_run-02_T1w_brain_anat_n4.nii.gz
Index 67: sub-OAS30111_ses-d0385_run-02_T1w_brain_anat_n4.nii.gz
Index 79: sub-OAS30296_ses-d0069

In [74]:
len(failed_registration_final_t1_dataset_oasis)

443

In [79]:
# Get the list of files from the OASIS T1 dataset
ixi_files = os.listdir(final_t1_dataset_ixi)
failed_registration_final_t1_dataset_ixi_filenames = []

# Print the filenames corresponding to the failed registration indexes
print(f"Failed registration files in ixi T1 dataset ({len(failed_registration_final_t1_dataset_ixi)} files):\n")

for idx in failed_registration_final_t1_dataset_ixi:
    if idx < len(ixi_files):
        print(f"Index {idx}: {ixi_files[idx]}")
        failed_registration_final_t1_dataset_ixi_filenames.append(ixi_files[idx])
    else:
        print(f"Index {idx}: OUT OF RANGE (total files: {len(ixi_files)})")



Failed registration files in ixi T1 dataset (3 files):

Index 50: IXI175-HH-1570-T1.nii.gz
Index 175: IXI639-Guys-1088-T1.nii.gz
Index 197: IXI489-Guys-1014-T1.nii.gz


In [80]:
# Append to CSV file
output_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_t1_dataset/exclude.csv")
with open(output_csv, "a", newline="") as f:
    writer = csv.writer(f)
    for filename in failed_registration_final_t1_dataset_ixi_filenames:
        writer.writerow(["datasets/final_t1_dataset/ixi_registered_final/" + filename])
print(f"\nWrote {len(failed_registration_final_t1_dataset_ixi_filenames)} entries to {output_csv}")


Wrote 3 entries to /bettik/PROJECTS/pr-gin5_aini/fehrdelt/AnoDiffExperiments/data_splits_lists/final_t1_dataset/exclude.csv


In [81]:
# Save to CSV file
output_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_t1_dataset/exclude.csv")
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    for filename in failed_registration_final_t1_dataset_oasis_filenames:
        writer.writerow(["datasets/final_t1_dataset/oasis_registered/" + filename])
print(f"\nWrote {len(failed_registration_final_t1_dataset_oasis_filenames)} entries to {output_csv}")


Wrote 446 entries to /bettik/PROJECTS/pr-gin5_aini/fehrdelt/AnoDiffExperiments/data_splits_lists/final_t1_dataset/exclude.csv


In [ ]:
# Display the first 4 failed registration files from OASIS T1 dataset
fig, axes = plt.subplots(6, 6, figsize=(10, 10))
axes = axes.flatten()

for i in range(min(36, len(failed_registration_final_t1_dataset_oasis_filenames))):
    filename = failed_registration_final_t1_dataset_oasis_filenames[i]
    img_path = os.path.join(final_t1_dataset_oasis, filename)
    
    # Load the NIFTI image
    img = nib.load(img_path)
    data = img.get_fdata()
    
    # Get the middle axial slice
    middle_slice_z = data.shape[2] // 2
    image_slice = data[:, :, middle_slice_z]
    
    # Display the slice
    axes[i].imshow(image_slice, cmap='gray')
    axes[i].set_title(f'{filename}', fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

hd-bet failed to completely remove the skull for sub-OAS30121_ses-d0768_run-04_T1w_brain_anat_n4.nii.gz and other images

Maybe because oasis images are defaced

In [ ]:
open 

In [41]:
print(os.listdir(final_t1_dataset_oasis)[554])

sub-OAS30121_ses-d0768_run-04_T1w_brain_anat_n4.nii.gz


retrieve the bad image names

In [ ]:
hist_norm_target_value = 200
max_display_value=450

# Display the failed registration images from T1 IXI dataset
filelist = os.listdir(final_t1_dataset_ixi)

for idx in failed_registration_final_t1_dataset_ixi:
    img_path = os.path.join(final_t1_dataset_ixi, filelist[int(idx)])
    print(img_path)
    img = nib.load(img_path)
    data = img.get_fdata()

    # Get middle axial slice
    mid_z = data.shape[2] // 2
    image_slice = data[:, :, mid_z]

    # Compute the histogram of the image slice
    hist, bins = np.histogram(image_slice.flatten(), bins=100, range=(np.max(image_slice)/5.0, np.max(image_slice)))
    # Find the value corresponding to the maximum of the histogram
    most_occurred_pixel_value = bins[np.argmax(hist)]
    image_slice = image_slice/most_occurred_pixel_value*hist_norm_target_value
    
    
    
    plt.figure(figsize=(3, 3))
    plt.imshow(image_slice, cmap='gray', vmin=0, vmax=max_display_value)
    plt.title(f'Index {idx}: {filelist[int(idx)]}')
    plt.axis('off')
    plt.show()

#### SOOP VISU

soop : y'a certaines images qui ont une meilleur résolution dans le sens coronal et pas axial (les slices epaisses sont dans un autre sens -> faire un script qui garde que les images qui on été acquises avec des grosses slices dans le sens axial ou c'est trop de la triche ?)

In [4]:
failed_registration_images_flair_soop = [50, 210, 270, 352, 466, 489, 496, 499, 523, 694, 734, 749, 773, 824, 870, 898, 924, 909, 993, 1172, 1400, 1413, 1456, 1505, 1533, 1548, 1564, 1584, 1590, 1604, 1619, 1632, 1669, 1698]
artefacted_images_flair_soop = [493, 794, 863, 1286, 1594, 1642]

failed_registration_images_adc_soop = [5, 108, 155, 186, 316, 364, 470, 481, 488, 531, 619, 672, 676, 680, 685, 758, 847, 874, 907, 959, 968, 1045, 1171, 1225, 1251, 1320, 1652, 1662]
artefacted_images_adc_soop = [358, 1195]
failed_histogram_normalization_images_adc_soop = [352, 857, 965, 1126, 1386]


In [8]:
final_soop_dataset_small_flair_filelist = glob.glob(final_soop_dataset_small_flair + "*.nii.gz")
final_soop_dataset_small_adc_filelist = glob.glob(final_soop_dataset_small_adc + "*.nii.gz")

print(f"failed registration flair soop: {[os.path.basename(final_soop_dataset_small_flair_filelist[i]).split('_')[0] for i in failed_registration_images_flair_soop]}")
print(f"artefaced images flair soop: {[os.path.basename(final_soop_dataset_small_flair_filelist[i]).split('_')[0] for i in artefacted_images_flair_soop]}")

print(f"failed registration adc soop: {[os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in failed_registration_images_adc_soop]}")
print(f"artefaced images adc soop: {[os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in artefacted_images_adc_soop]}")
print(f"failed histogram normalization adc soop: {[os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in failed_histogram_normalization_images_adc_soop]}")

failed_images = set()
failed_images.update([os.path.basename(final_soop_dataset_small_flair_filelist[i]).split('_')[0] for i in failed_registration_images_flair_soop])
failed_images.update([os.path.basename(final_soop_dataset_small_flair_filelist[i]).split('_')[0] for i in artefacted_images_flair_soop])
failed_images.update([os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in failed_registration_images_adc_soop])
failed_images.update([os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in artefacted_images_adc_soop])
failed_images.update([os.path.basename(final_soop_dataset_small_adc_filelist[i]).split('_')[0] for i in failed_histogram_normalization_images_adc_soop])

print(f"set of failed images soop: {failed_images}")

output_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_soop_dataset_small/exclude.csv")
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    for subj in sorted(failed_images):
        writer.writerow([subj])
print(f"Wrote {len(failed_images)} entries to {output_csv}")

failed registration flair soop: ['sub-1203', 'sub-1041', 'sub-279', 'sub-1727', 'sub-989', 'sub-984', 'sub-1558', 'sub-343', 'sub-707', 'sub-1261', 'sub-7', 'sub-605', 'sub-949', 'sub-234', 'sub-855', 'sub-1491', 'sub-185', 'sub-846', 'sub-560', 'sub-767', 'sub-1698', 'sub-617', 'sub-249', 'sub-1717', 'sub-1183', 'sub-512', 'sub-1138', 'sub-1308', 'sub-1660', 'sub-1610', 'sub-1251', 'sub-199', 'sub-887', 'sub-1119']
artefaced images flair soop: ['sub-1091', 'sub-148', 'sub-1112', 'sub-171', 'sub-1292', 'sub-1496']
failed registration adc soop: ['sub-208', 'sub-526', 'sub-767', 'sub-151', 'sub-1502', 'sub-256', 'sub-1175', 'sub-1307', 'sub-1590', 'sub-234', 'sub-52', 'sub-586', 'sub-40', 'sub-1133', 'sub-1592', 'sub-897', 'sub-1344', 'sub-1166', 'sub-764', 'sub-120', 'sub-1616', 'sub-1355', 'sub-34', 'sub-438', 'sub-1698', 'sub-1566', 'sub-1540', 'sub-188']
artefaced images adc soop: ['sub-1529', 'sub-571']
failed histogram normalization adc soop: ['sub-1284', 'sub-640', 'sub-785', 'sub

In [ ]:
display_middle_slice(final_soop_dataset_small_adc, axe=2, nb_images=49, page_number=0, figsize=25, normalize_from_histogram_peak=True, hist_norm_target_value=200, max_display_value=650)

#### BRATS visu

In [9]:
problematic_files = [70, 81, 98, 107, 141, 190, 204, 220, 335, 396, 412, 420, 423, 424, 427, 431, 433, 446, 477, 524, 586, 629, 633, 666, 733, 742]

In [ ]:
filelist = os.listdir(final_flair_dataset_small_oasis)

for file in problematic_files:
    print(filelist[file])
    """ img_path = os.path.join(final_flair_dataset_small_oasis, filelist[file])
    img = nib.load(img_path)
    data = img.get_fdata()
    mid_z = data.shape[2] // 2
    slice_img = data[:, :, mid_z]
    vmax = np.percentile(slice_img, 99)
    plt.figure(figsize=(4, 4))
    plt.imshow(slice_img, cmap='gray', vmin=0, vmax=vmax)
    plt.title(filelist[file])
    plt.axis('off')
    plt.show() """

In [8]:
for i,file in enumerate(os.listdir(final_flair_dataset_small_oasis)):
    if "sub-OAS30405_ses-d1883_FLAIR" in file:
        print(i, file)

654 sub-OAS30405_ses-d1883_FLAIR.nii.gz


In [18]:
display_middle_slice_z_with_histogram(folder_path=final_flair_dataset_small_brats, nb_images=25, page_number=0, figsize=30, normalize_from_histogram_peak=True, normalize_before_select_slice=True, hist_norm_target_value=200, max_display_value=700)

### Visualize with monai transforms

In [9]:

train_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_flair_dataset_small/train.csv")
train_images_path = []

with open(train_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):
        #print(line)
        train_images_path.append(ROOT_DIR+line[0])

val_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_flair_dataset_small/val.csv")
val_images_path = []

with open(val_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):

        val_images_path.append(ROOT_DIR+line[0])



#train_datalist = sorted(train_images_path)
train_datalist = train_images_path

#val_datalist = sorted(val_images_path)
val_datalist = val_images_path

#test_unhealthy_datalist = test_unhealthy_images_path

batch_size = 64 # 32
num_workers = 16 # 4*num_gpus, 4*world_size

train_transforms = transforms.Compose(
[
    transforms.LoadImage(image_only=True),
    transforms.EnsureChannelFirst(),
    transforms.RandAffine(prob=0.5, rotate_range=(0.10, 0.10, 0.10)),#+- 0.15 radians for each axis
    #transforms.EnsureType(device=device, track_meta=False),(didn't work error) # convert the data to Tensor without meta, move to GPU and cache to avoid CPU -> GPU sync in every epoch
    custom_transforms.Get2DSliceWithRandomOffset(axis=2, fixed_offset=0, range_offset=10),
    transforms.RandScaleCrop(roi_scale=0.9, max_roi_scale=1.1, random_size=True),
    transforms.ResizeWithPadOrCrop(spatial_size=(128, 128)),
    custom_transforms.ScaleIntensityFromHistogramPeak(target_value=200.0),
    transforms.ScaleIntensityRange(a_min=0.0, a_max=450.0, b_min=0.0, b_max=1.0, clip=True),
    transforms.RandFlip(prob=0.5, spatial_axis=0),
    custom_transforms.SetBackgroundToZero()
])

train_ds = CacheDataset(data=train_datalist[:batch_size], transform=train_transforms) #TODO datalist[:32]
train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
)

758it [00:00, 193858.68it/s]
95it [00:00, 59873.61it/s]
Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:09<00:00,  6.46it/s]


In [ ]:
# Get the first batch of images from the train_loader
batch = next(iter(train_loader))

# Extract the images from the batch
images = batch  # Assuming the first element in the batch is the image tensor

# Plot the first 16 images
fig, axes = plt.subplots(8, 8, figsize=(20, 20))
for i, ax in enumerate(axes.flatten()):
    #print(images.shape)
    ax.imshow(images[i, 0, :, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)  # Assuming images are single-channel
    ax.axis('off')
plt.tight_layout()
plt.show()

In [5]:
bad_dallas_flair = [236]
bad_dallas_flair_names = os.listdir(final_flair_dataset_small_dallas)[236]

bad_lemon_flair = [148]
bad_lemon_flair_names = os.listdir(final_flair_dataset_small_lemon)[148]

print("Bad Dallas FLAIR files names:", bad_dallas_flair_names)
print("Bad lemon FLAIR files names:", bad_lemon_flair_names)

Bad Dallas FLAIR files names: sub-4799_ses-wave1_acq-FLAIR_run-1_T2w.nii.gz
Bad lemon FLAIR files names: sub-032360_ses-01_acq-lowres_FLAIR.nii.gz


### Compare histograms from hcp, dallas and aini stroke

In [ ]:
bad_files = ["sub-3242_ses-wave2_ADC.nii.gz", 
            "aini-stroke-17579_425560_ADC_HR_DIFF_RESOLVE_3MM_FP_ADC.nii.gz",
            "aini-stroke-13607_424097_ADC_HR_DIFF_RESOLVE_3MM_FP_ADC.nii.gz"]

def display_mean_histogram(folder_path):
    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter out non-NIFTI files
    nifti_files = [f for f in files if f.endswith('.nii') or f.endswith('.nii.gz')]
    
    # Create a figure to display all middle slices
    plt.figure(figsize=(10, 10))

    data_sum = []
    skipped_files = 0

    for i, nifti_file in enumerate(tqdm(nifti_files)):
        if nifti_file not in bad_files:
            # Load the NIFTI image
            img_path = os.path.join(folder_path, nifti_file)
            img = nib.load(img_path)
            data = img.get_fdata()
            if len(data_sum) ==0:
                data_sum.append(data.flatten())
            else:
                data_sum[0] += data.flatten()
        else:
            print(f"Skipping bad file: {nifti_file}")
            skipped_files += 1
    
    mean_data = data_sum[0]/(len(nifti_files)-skipped_files)

    # Set the title of the figure
    plt.title(f'Mean histogram of all NIFTI files in {folder_path} (skipped {skipped_files} bad files)')
    plt.hist(mean_data[(mean_data>1) & (mean_data<=3000)], 100)
    # Show the plot
    plt.show()



In [ ]:
display_mean_histogram(final_flair_dataset_small_dallas)
display_mean_histogram(final_flair_dataset_small_lemon)
